# 03 — Modeling

This notebook trains a LightGBM regressor to predict `s_true` — the midpoint of
the feasible starting-inventory interval [L, U] — for each station×day. It then
evaluates the model using coverage (primary), conditional efficiency (secondary),
and RMSE (tertiary), and produces station-level KPI scatter plots and cumulative
KPI trend charts over the test period.

## Assumptions & Dependencies

- `02_feature_engineering.ipynb` must have been run and must have written
  `data/processed/train.parquet` and `data/processed/test.parquet`.
- Required packages: `lightgbm`, `pandas`, `numpy`, `matplotlib`, `scikit-learn`.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.models import train_lgbm, evaluate_coverage, coverage_summary

train_df = pd.read_parquet('../data/processed/train.parquet')
test_df  = pd.read_parquet('../data/processed/test.parquet')

print("Train:", train_df.shape, "| Test:", test_df.shape)

## Feature Selection

The model uses 13 features — 7 lag (_prev) features, 5 rolling (_roll7) features,
and station_id as a categorical. `events_prev` is also treated as categorical.
All features use only information available the day before the prediction date.

In [ ]:
FEATURES = [
    'min_start_inventory_prev',
    'max_start_inventory_prev',
    'station_capacity_day_prev',
    'temperature_prev',
    'events_prev',
    'trips_departed_prev',
    'trips_arrived_prev',
    'trips_departed_roll7',
    'trips_arrived_roll7',
    'temperature_roll7',
    'min_start_inventory_roll7',
    'max_start_inventory_roll7',
    'station_id',
]
CATEGORICAL_FEATURES = ['station_id', 'events_prev']
TARGET = 's_true'

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[FEATURES]

## Model Training

LightGBM is used because it handles categorical features natively, is fast on
tabular data, and produces well-calibrated predictions suitable for the coverage
evaluation.

In [ ]:
model = train_lgbm(X_train, y_train, CATEGORICAL_FEATURES)
test_df['s_hat'] = model.predict(X_test)

## Coverage and Efficiency Evaluation

A prediction is **covered** if the rounded prediction falls within [L, U]. Coverage
is the primary metric. Conditional efficiency measures how close the prediction is
to s_true relative to the width of [L, U], and is only computed for covered rows.

In [ ]:
test_df = evaluate_coverage(test_df, pred_col='s_hat')
print(coverage_summary(test_df))

## Station-Level KPI Scatter

Visualize per-station coverage rate vs mean conditional efficiency to identify
stations where the model struggles.

In [ ]:
station_kpi = test_df.groupby('station_id').agg(
    coverage_rate=('covered', 'mean'),
    mean_efficiency=('efficiency', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(station_kpi['coverage_rate'], station_kpi['mean_efficiency'], alpha=0.5, s=20)
ax.set_xlabel('Coverage Rate')
ax.set_ylabel('Mean Conditional Efficiency')
ax.set_title('Station-Level KPI Scatter — LightGBM Midpoint Model')
plt.tight_layout()
fig.savefig('../reports/figures/station_kpi_scatter.png', dpi=150)
plt.show()

## Cumulative KPI Trend Over Time

Track how cumulative coverage rate and mean efficiency evolve across the test
period to detect drift or seasonality effects.

In [ ]:
daily_kpi = test_df.groupby('trip_date').agg(
    coverage_rate=('covered', 'mean'),
    mean_efficiency=('efficiency', 'mean'),
).reset_index().sort_values('trip_date')

daily_kpi['cumulative_coverage'] = daily_kpi['coverage_rate'].expanding().mean()
daily_kpi['cumulative_efficiency'] = daily_kpi['mean_efficiency'].expanding().mean()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(daily_kpi['trip_date'], daily_kpi['cumulative_coverage'])
axes[0].set_ylabel('Cumulative Coverage Rate')
axes[1].plot(daily_kpi['trip_date'], daily_kpi['cumulative_efficiency'])
axes[1].set_ylabel('Cumulative Mean Efficiency')
axes[1].set_xlabel('Date')
fig.suptitle('Cumulative KPI Trend — LightGBM Midpoint Model')
plt.tight_layout()
fig.savefig('../reports/figures/cumulative_kpi_trend.png', dpi=150)
plt.show()

## Summary

**Outputs produced by this notebook:**
- `reports/figures/station_kpi_scatter.png` — per-station coverage vs efficiency
- `reports/figures/cumulative_kpi_trend.png` — KPI trend over the test period
- The fitted `model` object (held in memory for optional hand-off to notebook 04)

**What the next notebook expects:** `04_rebalancing_optimization.ipynb` reads
`data/processed/test.parquet` and needs the fitted model (either re-trained or
serialized). It expects columns L, U, s_true, station_id, latitude_start,
longitude_start, and station_capacity_day.